# Indoor views of a building

This page draws the indoor elements of one building in 3D with `osm.in3d`, the accessor that pyrosm3d adds to pyrosm's `OSM` object: the indoor view, a web page, and the walking network alone. It then draws frames that were read with pyrosm directly.

The building is Maunula-talo, a community centre in Helsinki whose rooms, doors and stairwells are mapped on three levels. The repository holds a small extract of it for its tests, `tests/data/maunula.osm.pbf`. With a PBF file of your own, give `OSM` a `bounding_box` around the building you want.

## Draw the indoor view

pyrosm reads the file, and `osm.in3d` holds the views of the `OSM` object. `osm.in3d.plot_indoor()` reads the indoor elements, the walking network and the buildings with their parts, and returns a view, a `pyrosm3d.View`. In a notebook the view is the output of the cell: a page drawn with deck.gl, which it loads from the internet. It opens in the indoor view, seen from the south-east and above. One finger, or a drag with the mouse, moves the view; a pinch or the mouse wheel zooms; the Tilt and Turn sliders tilt and turn it. "Level" shows one level alone, and a tap on an element says what it is and lists its OpenStreetMap tags, apart from contact details such as e-mail addresses and phone numbers. The notes of the page, under "What this is and how to move", explain every control.

- Floors are plates coloured by the kind of area, 4 m apart above the ground floor and 6 m apart below it, and drawn taller than to scale until "True heights" is switched on. An area whose only level lies between two floors, such as a mezzanine tagged `level=0.5`, is drawn at its own height and has its own entry in "Level".
- Walls stand along the outlines of rooms and are cut away at 1.6 m, so that the rooms stay visible from above. A door leaves a gap in the wall.
- Stairwells and lift shafts are columns through the floors that they join.
- The walking network is drawn on its levels, in the colours of `plot_network`.
- Outer walls, doors and shops are off at the start; "Layers" switches them on.

"Buildings" shows the same elements in a solid model, with the buildings around them, and "See through" shows them as a map.

In [1]:
from pyrosm import OSM

osm = OSM("../tests/data/maunula.osm.pbf")
osm.in3d.plot_indoor()

## Open the view on one level

With `level`, the indoor view opens on that level, and "Level" still lists the others. The columns of stairwells and lifts are drawn with all levels only, since they pass through several.

In [2]:
osm.in3d.plot_indoor(level=1)

## Write a web page

`osm.in3d.to_html` writes the view as a page of its own, to open in a browser or to share. "Level" names the number of areas on each level, and a tap on a room, a line of the network, a door, a shop or the column of a stairwell or a lift shows what the element is, its levels, its OpenStreetMap tags apart from contact details, and a link to the element on openstreetmap.org. The page works on a phone as well: one finger moves the view, a pinch zooms and turns it, and three fingers tilt it.

The page loads deck.gl from the jsDelivr network, so it needs an internet connection; `deck_gl` gives the address of another copy, such as a file beside the page.

In [3]:
osm.in3d.to_html("maunula-talo.html", title="Maunula-talo")

PosixPath('maunula-talo.html')

## Draw the walking network alone

`osm.in3d.plot_network()` reads the walking network with `indoor_elements=True` and draws each edge at its levels, true to scale: levels lie 4 m apart above the ground floor and 6 m apart below it, over a flat ground at street level. A tap on an edge shows its name, its element and its levels. With `center` and `radius` it draws only what lies near a point, with `route` a path through the network, and with `graph` a networkx or igraph graph of `get_graph(indoor_elements=True)`, such as the one a route was found in. With `buildings=True` it also reads the buildings and draws them.

In this extract the streets and platforms are at street level, and the lines across the corridors of Maunula-talo are on levels 1 and 2.

In [4]:
osm.in3d.plot_network(buildings=True)

## Draw frames that you read yourself

The views draw frames that pyrosm reads: the indoor elements with the walking areas, the walking network with `indoor_elements=True`, and the buildings. The functions `pyrosm3d.plot_indoor` and `pyrosm3d.to_html` take these frames, so you can read them yourself, change them and draw the result. `get_indoor_elements` lists the levels of each element in the column `levels`: the lift in the first row has `level=0-2` and is on three levels.

In [5]:
elements = osm.get_indoor_elements(walking_areas=True)
on_several = elements[elements["levels"].str.contains(";", na=False)]
on_several[["id", "osm_type", "indoor", "highway", "level", "levels"]].head()

,id,osm_type,indoor,highway,level,levels
176,1568797319,way,room,elevator,0-2,0;1;2
178,1576010525,way,room,NaN,1;2,1;2
192,1576010636,way,room,NaN,1;2,1;2


A polygon among the elements is drawn as a floor when it has an `indoor` tag or is a platform, a lift area or a walking area, such as a square; other areas that lines of the walking network cross are floors too. A platform or a walking area without a level takes the level of its lines in the walking network, else its layer when that is below ground, else street level: the bus and tram platforms and the two squares beside Maunula-talo have no `level` tag and are on street level. Walls mapped as lines, doors, and the shops and amenities that have a level are drawn from the same frame.

Here the view is drawn from the elements alone, without the walking network, and opens on the second floor.

In [6]:
import pyrosm3d

pyrosm3d.plot_indoor(elements, buildings=osm.get_buildings(), level=2)